In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import mean_absolute_error, mean_squared_error
from pathlib import Path
import matplotlib.patches as mpatches

## $\text{\textcolor{green}{Setting Up Right csv}}$

In [ ]:
# =========================================================
# 0.5 AGGREGATE POWER COLUMNS
# =========================================================

def aggregate_power_columns(
    df_input,
    pv_cols=None,
    wind_power_cols=None,
    drop_original=False
):
    """
    Aggregate multiple PV and wind power columns into total signals.
    """
    df_output = df_input.copy()

    if pv_cols is not None:
        missing_pv = [col for col in pv_cols if col not in df_output.columns]
        if missing_pv:
            raise ValueError(f"Missing PV columns: {missing_pv}")

        df_output["PV_total"] = df_output[pv_cols].sum(axis=1, min_count=1)

    if wind_power_cols is not None:
        missing_wind = [col for col in wind_power_cols if col not in df_output.columns]
        if missing_wind:
            raise ValueError(f"Missing wind columns: {missing_wind}")

        df_output["Wind_total"] = df_output[wind_power_cols].sum(axis=1, min_count=1)

    if drop_original:
        cols_to_drop = []
        if pv_cols is not None:
            cols_to_drop.extend(pv_cols)
        if wind_power_cols is not None:
            cols_to_drop.extend(wind_power_cols)

        df_output = df_output.drop(columns=cols_to_drop)

    return df_output

## $\text{\textcolor{green}{Creation of Daily Profiles––Weather Input and Separate Power Outputs (PV and Wind)}}$

In [ ]:
# =========================================================
# 1. PREPARE DAILY PROFILES ONCE
# =========================================================

def prepare_daily_profiles_single_output(
    df_input,
    feature_cols,
    target_col,
    samples_per_day=48
):
    """
    Validate and reshape half-hourly data into daily profiles.

    Returns
    -------
    X_daily : DataFrame
        One row per valid day, containing all daily weather values.
    y_daily : DataFrame
        One row per valid day, containing the full-day target profile.
    valid_days : np.ndarray
        Sorted array of valid day labels.
    """
    required_cols = feature_cols + [target_col]

    df_work = df_input.copy().sort_index()
    df_work["date"] = df_work.index.date
    df_work["time"] = df_work.index.strftime("%H:%M")

    rows_per_day = df_work.groupby("date").size()
    full_days = rows_per_day[rows_per_day == samples_per_day].index
    df_work = df_work[df_work["date"].isin(full_days)].copy()

    valid_day_mask = (
        df_work.groupby("date")[required_cols]
        .apply(lambda day_data: not day_data.isna().any().any())
    )
    valid_days = np.array(sorted(valid_day_mask[valid_day_mask].index))
    df_work = df_work[df_work["date"].isin(valid_days)].copy()

    daily_feature_blocks = []

    for feature_name in feature_cols:
        daily_feature_matrix = df_work.pivot(
            index="date",
            columns="time",
            values=feature_name
        )
        daily_feature_matrix = daily_feature_matrix.reindex(
            sorted(daily_feature_matrix.columns),
            axis=1
        )
        daily_feature_matrix.columns = [
            f"{feature_name}_{time_label}"
            for time_label in daily_feature_matrix.columns
        ]
        daily_feature_blocks.append(daily_feature_matrix)

    X_daily = pd.concat(daily_feature_blocks, axis=1)

    y_daily = df_work.pivot(
        index="date",
        columns="time",
        values=target_col
    )
    y_daily = y_daily.reindex(sorted(y_daily.columns), axis=1)
    y_daily.columns = [
        f"{target_col}_{time_label}"
        for time_label in y_daily.columns
    ]

    common_days = X_daily.index.intersection(y_daily.index)
    X_daily = X_daily.loc[common_days]
    y_daily = y_daily.loc[common_days]
    valid_days = np.array(sorted(common_days))

    return X_daily, y_daily, valid_days

## $\text{\textcolor{green}{Choosing an Optimal K for KMeans}}$

### $\text{\textcolor{green}{Create Folds}}$

In [ ]:
# =========================================================
# 2. CREATE MONTHLY SAMPLED-DAY FOLDS
# =========================================================

def make_monthly_sampled_folds_from_days(
    valid_days,
    n_folds=5,
    test_days_per_month=3,
    random_state=42
):
    """
    Create folds from an existing set of valid days.
    """
    valid_day_table = pd.DataFrame({"date": pd.to_datetime(valid_days)})
    valid_day_table["month"] = valid_day_table["date"].dt.to_period("M")

    rng = np.random.RandomState(random_state)
    folds = []

    for fold_id in range(n_folds):
        sampled_test_days = []

        for _, month_group in valid_day_table.groupby("month"):
            month_days = month_group["date"].dt.date.values
            n_test_days = min(test_days_per_month, len(month_days))

            chosen_test_days = rng.choice(
                month_days,
                size=n_test_days,
                replace=False
            )
            sampled_test_days.extend(chosen_test_days.tolist())

        test_days = np.array(sorted(set(sampled_test_days)))
        train_days = np.array(sorted(set(valid_days) - set(test_days)))

        folds.append({
            "fold_id": fold_id,
            "train_days": train_days,
            "test_days": test_days
        })

    return folds

### $\text{\textcolor{green}{Performance Metrics \& Band for KMeans}}$

In [ ]:
# =========================================================
# 3. METRIC HELPERS
# =========================================================

def compute_profile_metrics(y_true_daily, y_pred_daily):
    """
    Compute error metrics for daily profiles.
    """
    y_true_flat = y_true_daily.to_numpy().flatten()
    y_pred_flat = y_pred_daily.to_numpy().flatten()

    mae = mean_absolute_error(y_true_flat, y_pred_flat)
    rmse = np.sqrt(mean_squared_error(y_true_flat, y_pred_flat))
    daily_mae = np.mean(
        np.abs(y_true_daily.to_numpy() - y_pred_daily.to_numpy()).mean(axis=1)
    )

    return {
        "mae": float(mae),
        "rmse": float(rmse),
        "daily_mae": float(daily_mae)
    }


def compute_band_coverage(y_true_daily, y_lower_daily, y_upper_daily):
    """
    Compute coverage of the uncertainty band.
    """
    inside_band = (y_true_daily >= y_lower_daily) & (y_true_daily <= y_upper_daily)

    point_coverage = inside_band.to_numpy().mean()
    mean_daily_coverage = inside_band.mean(axis=1).mean()

    return {
        "point_coverage": float(point_coverage),
        "mean_daily_coverage": float(mean_daily_coverage)
    }

### $\text{\textcolor{green}{Evaluate K for a single generation source}}$

In [ ]:
# =========================================================
# 4. EVALUATE ONE K FOR A SINGLE TARGET
# =========================================================

def evaluate_kmeans_for_k_single_output(
    X_train_daily,
    y_train_daily,
    X_test_daily,
    y_test_daily,
    k,
    random_state=42
):
    """
    Train one KMeans model on weather-day profiles and predict one
    daily power target from the cluster assignments.

    Also returns a 10th-90th percentile uncertainty band.
    """
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_daily)
    X_test_scaled = scaler.transform(X_test_daily)

    kmeans_model = KMeans(
        n_clusters=k,
        random_state=random_state,
        n_init=10
    )

    train_cluster_labels = kmeans_model.fit_predict(X_train_scaled)
    test_cluster_labels = kmeans_model.predict(X_test_scaled)

    cluster_size_series = pd.Series(train_cluster_labels).value_counts().sort_index()

    cluster_size_info = {
        "cluster_sizes": cluster_size_series,
        "min_cluster_size": int(cluster_size_series.min()),
        "max_cluster_size": int(cluster_size_series.max()),
        "mean_cluster_size": float(cluster_size_series.mean()),
        "median_cluster_size": float(cluster_size_series.median()),
        "n_small_clusters_lt_5": int((cluster_size_series < 5).sum()),
        "n_small_clusters_lt_10": int((cluster_size_series < 10).sum())
    }

    train_targets_with_clusters = y_train_daily.copy()
    train_targets_with_clusters["cluster"] = train_cluster_labels

    cluster_profile_mean = train_targets_with_clusters.groupby("cluster").mean()
    cluster_profile_q10 = train_targets_with_clusters.groupby("cluster").quantile(0.10)
    cluster_profile_q90 = train_targets_with_clusters.groupby("cluster").quantile(0.90)

    y_pred_daily = pd.DataFrame(
        index=y_test_daily.index,
        columns=y_test_daily.columns,
        dtype=float
    )
    y_pred_q10_daily = pd.DataFrame(
        index=y_test_daily.index,
        columns=y_test_daily.columns,
        dtype=float
    )
    y_pred_q90_daily = pd.DataFrame(
        index=y_test_daily.index,
        columns=y_test_daily.columns,
        dtype=float
    )

    for day, cluster_label in zip(y_test_daily.index, test_cluster_labels):
        y_pred_daily.loc[day] = cluster_profile_mean.loc[cluster_label].values
        y_pred_q10_daily.loc[day] = cluster_profile_q10.loc[cluster_label].values
        y_pred_q90_daily.loc[day] = cluster_profile_q90.loc[cluster_label].values

    point_metrics = compute_profile_metrics(y_test_daily, y_pred_daily)
    coverage_metrics = compute_band_coverage(
        y_true_daily=y_test_daily,
        y_lower_daily=y_pred_q10_daily,
        y_upper_daily=y_pred_q90_daily
    )

    metrics = {**point_metrics, **coverage_metrics}

    return {
        "k": k,
        "scaler": scaler,
        "kmeans_model": kmeans_model,
        "train_cluster_labels": train_cluster_labels,
        "test_cluster_labels": test_cluster_labels,
        "cluster_profile_mean": cluster_profile_mean,
        "cluster_profile_q10": cluster_profile_q10,
        "cluster_profile_q90": cluster_profile_q90,
        "y_pred_daily": y_pred_daily,
        "y_pred_q10_daily": y_pred_q10_daily,
        "y_pred_q90_daily": y_pred_q90_daily,
        "metrics": metrics,
        "cluster_size_info": cluster_size_info
    }

# =========================================================
# 5. EVALUATE ONE K ACROSS FOLDS
# =========================================================

def evaluate_k_across_folds_single_output(
    X_daily,
    y_daily,
    folds,
    k,
    random_state=42
):
    """
    Evaluate one candidate K across fixed folds for one target.
    """
    fold_metrics = []

    for fold_definition in folds:
        train_days = fold_definition["train_days"]
        test_days = fold_definition["test_days"]

        X_train_daily = X_daily.loc[train_days]
        y_train_daily = y_daily.loc[train_days]

        X_test_daily = X_daily.loc[test_days]
        y_test_daily = y_daily.loc[test_days]

        fold_result = evaluate_kmeans_for_k_single_output(
            X_train_daily=X_train_daily,
            y_train_daily=y_train_daily,
            X_test_daily=X_test_daily,
            y_test_daily=y_test_daily,
            k=k,
            random_state=random_state
        )

        cluster_size_info = fold_result["cluster_size_info"]
        metrics = fold_result["metrics"]

        fold_metrics.append({
            "k": k,
            "fold_id": fold_definition["fold_id"],
            "n_train_days": len(train_days),
            "n_test_days": len(test_days),
            "mae": metrics["mae"],
            "rmse": metrics["rmse"],
            "daily_mae": metrics["daily_mae"],
            "point_coverage": metrics["point_coverage"],
            "mean_daily_coverage": metrics["mean_daily_coverage"],
            "min_cluster_size": cluster_size_info["min_cluster_size"],
            "max_cluster_size": cluster_size_info["max_cluster_size"],
            "mean_cluster_size": cluster_size_info["mean_cluster_size"],
            "median_cluster_size": cluster_size_info["median_cluster_size"],
            "n_small_clusters_lt_5": cluster_size_info["n_small_clusters_lt_5"],
            "n_small_clusters_lt_10": cluster_size_info["n_small_clusters_lt_10"],
            "min_cluster_fraction": cluster_size_info["min_cluster_size"] / len(train_days)
        })

    fold_results_df = pd.DataFrame(fold_metrics)

    summary = {
        "k": k,
        "mean_mae": fold_results_df["mae"].mean(),
        "std_mae": fold_results_df["mae"].std(ddof=1),
        "mean_rmse": fold_results_df["rmse"].mean(),
        "std_rmse": fold_results_df["rmse"].std(ddof=1),
        "mean_daily_mae": fold_results_df["daily_mae"].mean(),
        "std_daily_mae": fold_results_df["daily_mae"].std(ddof=1),
        "mean_point_coverage": fold_results_df["point_coverage"].mean(),
        "std_point_coverage": fold_results_df["point_coverage"].std(ddof=1),
        "mean_mean_daily_coverage": fold_results_df["mean_daily_coverage"].mean(),
        "std_mean_daily_coverage": fold_results_df["mean_daily_coverage"].std(ddof=1),
        "mean_min_cluster_size": fold_results_df["min_cluster_size"].mean(),
        "worst_min_cluster_size": fold_results_df["min_cluster_size"].min(),
        "mean_max_cluster_size": fold_results_df["max_cluster_size"].mean(),
        "mean_mean_cluster_size": fold_results_df["mean_cluster_size"].mean(),
        "mean_median_cluster_size": fold_results_df["median_cluster_size"].mean(),
        "mean_n_small_clusters_lt_5": fold_results_df["n_small_clusters_lt_5"].mean(),
        "mean_n_small_clusters_lt_10": fold_results_df["n_small_clusters_lt_10"].mean(),
        "mean_min_cluster_fraction": fold_results_df["min_cluster_fraction"].mean(),
        "worst_min_cluster_fraction": fold_results_df["min_cluster_fraction"].min()
    }

    return fold_results_df, summary


# =========================================================
# 6. SEARCH BEST K FOR A SINGLE TARGET
# =========================================================

def search_best_k_with_folds_single_output(
    X_daily,
    y_daily,
    valid_days,
    k_values,
    n_folds=5,
    test_days_per_month=3,
    fold_random_state=42,
    model_random_state=42,
    min_allowed_cluster_size=5
):
    """
    Generate folds once from valid_days, reuse them across all K values,
    and choose best K from fold performance with a cluster-size constraint.
    """
    folds = make_monthly_sampled_folds_from_days(
        valid_days=valid_days,
        n_folds=n_folds,
        test_days_per_month=test_days_per_month,
        random_state=fold_random_state
    )

    all_fold_results = []
    k_summary_rows = []

    for k in k_values:
        fold_results_df, k_summary = evaluate_k_across_folds_single_output(
            X_daily=X_daily,
            y_daily=y_daily,
            folds=folds,
            k=k,
            random_state=model_random_state
        )

        all_fold_results.append(fold_results_df)
        k_summary_rows.append(k_summary)

    all_fold_results_df = pd.concat(all_fold_results, ignore_index=True)
    summary_df = pd.DataFrame(k_summary_rows).sort_values("k").reset_index(drop=True)

    eligible_summary_df = summary_df[
        summary_df["worst_min_cluster_size"] >= min_allowed_cluster_size
    ].copy()

    if len(eligible_summary_df) == 0:
        raise ValueError(
            f"No K values satisfy worst_min_cluster_size >= {min_allowed_cluster_size}. "
            "Try smaller K values or relax the threshold."
        )

    best_k_row = eligible_summary_df.loc[
        eligible_summary_df["mean_daily_mae"].idxmin()
    ]
    best_k = int(best_k_row["k"])

    return {
        "folds": folds,
        "all_fold_results": all_fold_results_df,
        "summary": summary_df,
        "eligible_summary": eligible_summary_df,
        "best_k": best_k
    }

## $\text{\textcolor{green}{Fitting Model to Optimal K}}$

### $\text{\textcolor{green}{Final Train/Test Split}}$

In [ ]:
# =========================================================
# 7. CREATE FRESH FINAL SPLIT
# =========================================================

def create_final_monthly_split_from_days(
    valid_days,
    test_days_per_month=3,
    random_state=999
):
    """
    Create one fresh final split from valid_days.
    """
    valid_day_table = pd.DataFrame({"date": pd.to_datetime(valid_days)})
    valid_day_table["month"] = valid_day_table["date"].dt.to_period("M")

    rng = np.random.RandomState(random_state)
    sampled_final_test_days = []

    for _, month_group in valid_day_table.groupby("month"):
        month_days = month_group["date"].dt.date.values
        n_test_days = min(test_days_per_month, len(month_days))

        chosen_test_days = rng.choice(
            month_days,
            size=n_test_days,
            replace=False
        )
        sampled_final_test_days.extend(chosen_test_days.tolist())

    final_test_days = np.array(sorted(set(sampled_final_test_days)))
    final_train_days = np.array(sorted(set(valid_days) - set(final_test_days)))

    return {
        "train_days": final_train_days,
        "test_days": final_test_days
    }

### $\text{\textcolor{green}{Fit Model with Optimal K to Single Generation Source}}$

In [ ]:
# =========================================================
# 8. FINAL MODEL FIT FOR A SINGLE TARGET
# =========================================================

def fit_final_model_single_output(
    X_daily,
    y_daily,
    valid_days,
    best_k,
    final_test_days_per_month=3,
    final_split_random_state=999,
    model_random_state=42
):
    """
    After best K is selected:
    - create a fresh final split
    - fit clusters on final training days
    - evaluate both:
        * in-sample performance on train days
        * out-of-sample performance on test days

    Returns everything needed for:
    - final evaluation
    - cluster interpretation using TRAIN data only
    """
    final_split = create_final_monthly_split_from_days(
        valid_days=valid_days,
        test_days_per_month=final_test_days_per_month,
        random_state=final_split_random_state
    )

    final_train_days = final_split["train_days"]
    final_test_days = final_split["test_days"]

    X_train_daily = X_daily.loc[final_train_days]
    y_train_daily = y_daily.loc[final_train_days]

    X_test_daily = X_daily.loc[final_test_days]
    y_test_daily = y_daily.loc[final_test_days]

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_daily)
    X_test_scaled = scaler.transform(X_test_daily)

    kmeans_model = KMeans(
        n_clusters=best_k,
        random_state=model_random_state,
        n_init=10
    )

    train_cluster_labels = kmeans_model.fit_predict(X_train_scaled)
    test_cluster_labels = kmeans_model.predict(X_test_scaled)

    cluster_size_series = pd.Series(train_cluster_labels).value_counts().sort_index()

    cluster_size_info = {
        "cluster_sizes": cluster_size_series,
        "min_cluster_size": int(cluster_size_series.min()),
        "max_cluster_size": int(cluster_size_series.max()),
        "mean_cluster_size": float(cluster_size_series.mean()),
        "median_cluster_size": float(cluster_size_series.median()),
        "n_small_clusters_lt_5": int((cluster_size_series < 5).sum()),
        "n_small_clusters_lt_10": int((cluster_size_series < 10).sum())
    }

    train_targets_with_clusters = y_train_daily.copy()
    train_targets_with_clusters["cluster"] = train_cluster_labels

    cluster_profile_mean = train_targets_with_clusters.groupby("cluster").mean()
    cluster_profile_q10 = train_targets_with_clusters.groupby("cluster").quantile(0.10)
    cluster_profile_q90 = train_targets_with_clusters.groupby("cluster").quantile(0.90)

    # TRAIN predictions
    y_train_pred_daily = pd.DataFrame(
        index=y_train_daily.index,
        columns=y_train_daily.columns,
        dtype=float
    )
    y_train_pred_q10_daily = pd.DataFrame(
        index=y_train_daily.index,
        columns=y_train_daily.columns,
        dtype=float
    )
    y_train_pred_q90_daily = pd.DataFrame(
        index=y_train_daily.index,
        columns=y_train_daily.columns,
        dtype=float
    )

    for day, cluster_label in zip(y_train_daily.index, train_cluster_labels):
        y_train_pred_daily.loc[day] = cluster_profile_mean.loc[cluster_label].values
        y_train_pred_q10_daily.loc[day] = cluster_profile_q10.loc[cluster_label].values
        y_train_pred_q90_daily.loc[day] = cluster_profile_q90.loc[cluster_label].values

    train_metrics = {
        **compute_profile_metrics(y_train_daily, y_train_pred_daily),
        **compute_band_coverage(
            y_true_daily=y_train_daily,
            y_lower_daily=y_train_pred_q10_daily,
            y_upper_daily=y_train_pred_q90_daily
        )
    }

    # TEST predictions
    y_test_pred_daily = pd.DataFrame(
        index=y_test_daily.index,
        columns=y_test_daily.columns,
        dtype=float
    )
    y_test_pred_q10_daily = pd.DataFrame(
        index=y_test_daily.index,
        columns=y_test_daily.columns,
        dtype=float
    )
    y_test_pred_q90_daily = pd.DataFrame(
        index=y_test_daily.index,
        columns=y_test_daily.columns,
        dtype=float
    )

    for day, cluster_label in zip(y_test_daily.index, test_cluster_labels):
        y_test_pred_daily.loc[day] = cluster_profile_mean.loc[cluster_label].values
        y_test_pred_q10_daily.loc[day] = cluster_profile_q10.loc[cluster_label].values
        y_test_pred_q90_daily.loc[day] = cluster_profile_q90.loc[cluster_label].values

    test_metrics = {
        **compute_profile_metrics(y_test_daily, y_test_pred_daily),
        **compute_band_coverage(
            y_true_daily=y_test_daily,
            y_lower_daily=y_test_pred_q10_daily,
            y_upper_daily=y_test_pred_q90_daily
        )
    }

    return {
        "best_k": best_k,
        "final_train_days": final_train_days,
        "final_test_days": final_test_days,
        "X_train_daily": X_train_daily,
        "y_train_daily": y_train_daily,
        "X_test_daily": X_test_daily,
        "y_test_daily": y_test_daily,
        "model_result": {
            "scaler": scaler,
            "kmeans_model": kmeans_model,
            "train_cluster_labels": train_cluster_labels,
            "test_cluster_labels": test_cluster_labels,
            "cluster_profile_mean": cluster_profile_mean,
            "cluster_profile_q10": cluster_profile_q10,
            "cluster_profile_q90": cluster_profile_q90,
            "y_train_pred_daily": y_train_pred_daily,
            "y_train_pred_q10_daily": y_train_pred_q10_daily,
            "y_train_pred_q90_daily": y_train_pred_q90_daily,
            "y_test_pred_daily": y_test_pred_daily,
            "y_test_pred_q10_daily": y_test_pred_q10_daily,
            "y_test_pred_q90_daily": y_test_pred_q90_daily,
            "train_metrics": train_metrics,
            "test_metrics": test_metrics,
            "cluster_size_info": cluster_size_info
        }
    }

## $\text{\textcolor{green}{Plots Functions}}$

In [ ]:
# 9. PLOTTING HELPERS
# =========================================================

def make_half_hour_time_labels():
    return [f"{hour:02d}:{minute:02d}" for hour in range(24) for minute in (0, 30)]


def plot_mae_vs_k(summary_df, target_name):
    plt.figure(figsize=(8, 5))
    plt.plot(summary_df["k"], summary_df["mean_daily_mae"], marker="o", label=f"{target_name} daily MAE")
    plt.xlabel("Number of clusters (K)")
    plt.ylabel("Mean daily MAE across folds")
    plt.title(f"MAE vs K ({target_name})")
    plt.legend()
    plt.grid(True)
    plt.show()


def plot_min_cluster_size_vs_k(summary_df):
    plt.figure(figsize=(8, 5))
    plt.plot(
        summary_df["k"],
        100 * summary_df["mean_min_cluster_fraction"],
        marker="o",
        label="Mean min cluster size (%)"
    )
    plt.plot(
        summary_df["k"],
        100 * summary_df["worst_min_cluster_fraction"],
        marker="o",
        label="Worst min cluster size (%)"
    )
    plt.xlabel("Number of clusters (K)")
    plt.ylabel("Minimum cluster size (% of training set)")
    plt.title("Minimum cluster size vs K")
    plt.legend()
    plt.grid(True)
    plt.show()


def plot_coverage_vs_k(summary_df, target_name):
    plt.figure(figsize=(8, 5))
    plt.plot(
        summary_df["k"],
        100 * summary_df["mean_point_coverage"],
        marker="o",
        label=f"{target_name} point coverage"
    )
    plt.axhline(80, linestyle="--", label="Nominal 80% target")
    plt.xlabel("Number of clusters (K)")
    plt.ylabel("Coverage (%)")
    plt.title(f"10th-90th percentile band coverage vs K ({target_name})")
    plt.legend()
    plt.grid(True)
    plt.show()


def plot_cluster_spread_from_final_model(
    final_result,
    wind_feature="windspeed",
    radiation_feature="radia_glob"
):
    """
    Plot cluster spread using TRAIN data only from the final model.
    """
    X_train_daily = final_result["X_train_daily"]

    cluster_labels = pd.Series(
        final_result["model_result"]["train_cluster_labels"],
        index=X_train_daily.index,
        name="cluster"
    )

    wind_cols = [col for col in X_train_daily.columns if col.startswith(f"{wind_feature}_")]
    radiation_cols = [col for col in X_train_daily.columns if col.startswith(f"{radiation_feature}_")]

    if len(wind_cols) == 0:
        raise ValueError(f"No columns found for wind feature '{wind_feature}'")
    if len(radiation_cols) == 0:
        raise ValueError(f"No columns found for radiation feature '{radiation_feature}'")

    plot_df = pd.DataFrame({
        "daily_mean_windspeed": X_train_daily[wind_cols].mean(axis=1),
        "daily_mean_radiation": X_train_daily[radiation_cols].mean(axis=1),
        "cluster": cluster_labels
    }, index=X_train_daily.index)

    plt.figure(figsize=(8, 6))

    for cluster_id in sorted(plot_df["cluster"].unique()):
        cluster_data = plot_df[plot_df["cluster"] == cluster_id]
        plt.scatter(
            cluster_data["daily_mean_radiation"],
            cluster_data["daily_mean_windspeed"],
            label=f"Cluster {cluster_id}",
            alpha=0.7
        )

    plt.xlabel(f"Daily mean {radiation_feature}")
    plt.ylabel(f"Daily mean {wind_feature}")
    plt.title("Cluster spread (TRAIN data only)")
    plt.legend()
    plt.grid(True)
    plt.show()


def plot_cluster_profiles_from_final_model(final_result, target_name, show_band = True):
    """
    Plot average cluster profiles learned from TRAIN data only.
    show_band : bool
        If True, plot 10th-90th percentile band
        If False, plot only the mean profile
    """
    cluster_profile_mean = final_result["model_result"]["cluster_profile_mean"]

    # Only needed if bands are shown
    if show_band:
        cluster_profile_q10 = final_result["model_result"]["cluster_profile_q10"]
        cluster_profile_q90 = final_result["model_result"]["cluster_profile_q90"]

    time_labels = [f"{h:02d}:{m:02d}" for h in range(24) for m in (0, 30)]
    x = np.arange(48)
    tick_step = 4

    plt.figure(figsize=(12, 5))

    for cluster_id in cluster_profile_mean.index:
        mean_values = cluster_profile_mean.loc[cluster_id].values

        plt.plot(
            x,
            mean_values,
            label=f"Cluster {cluster_id}"
        )

        # Optional uncertainty band
        if show_band:
            plt.fill_between(
                x,
                cluster_profile_q10.loc[cluster_id].values,
                cluster_profile_q90.loc[cluster_id].values,
                alpha=0.2
            )

    plt.xticks(x[::tick_step], time_labels[::tick_step], rotation=45)
    plt.xlabel("Time of day")
    plt.ylabel(target_name)

    if show_band:
        plt.title(f"{target_name} cluster profiles (with 10–90% band)")
    else:
        plt.title(f"{target_name} cluster profiles (mean only)")

    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()



def plot_true_vs_predicted_month(
    final_result,
    target_name,
    month,
    true_color="#1b7f3b",       # green
    pred_color="#c51b7d",       # pink
    train_bg="#f7f7f7",         # very light grey
    test_bg="#fde6ef"           # very light pink
):
    """
    Plot one TRUE curve and one PREDICTED curve across all available days
    in a given month, while indicating whether each day is in-sample
    (train) or out-of-sample (test) through background shading.

    Parameters
    ----------
    final_result : dict
        Output from fit_final_model_single_output(...)
    target_name : str
        e.g. "PV_total" or "Wind_total"
    month : str
        Format "YYYY-MM", e.g. "2025-05"
    """
    model_result = final_result["model_result"]

    y_train_true = final_result["y_train_daily"]
    y_train_pred = model_result["y_train_pred_daily"]
    y_test_true = final_result["y_test_daily"]
    y_test_pred = model_result["y_test_pred_daily"]

    month_period = pd.Period(month, freq="M")

    train_days = sorted([
        d for d in y_train_true.index
        if pd.Period(str(d), freq="M") == month_period
    ])
    test_days = sorted([
        d for d in y_test_true.index
        if pd.Period(str(d), freq="M") == month_period
    ])

    if len(train_days) == 0 and len(test_days) == 0:
        raise ValueError(f"No data available for month {month}")

    combined_days = sorted(train_days + test_days)

    true_vals = []
    pred_vals = []
    sample_type = []

    for d in combined_days:
        if d in train_days:
            true_vals.extend(y_train_true.loc[d].values)
            pred_vals.extend(y_train_pred.loc[d].values)
            sample_type.extend(["train"] * 48)
        else:
            true_vals.extend(y_test_true.loc[d].values)
            pred_vals.extend(y_test_pred.loc[d].values)
            sample_type.extend(["test"] * 48)

    true_vals = np.array(true_vals, dtype=float)
    pred_vals = np.array(pred_vals, dtype=float)
    sample_type = np.array(sample_type)

    x = np.arange(len(true_vals))

    day_starts = np.arange(0, 48 * len(combined_days), 48)
    day_labels = [pd.Timestamp(d).strftime("%Y-%m-%d") for d in combined_days]

    fig, ax = plt.subplots(figsize=(15, 6))

    # Background shading by day
    for i, d in enumerate(combined_days):
        x0 = i * 48
        x1 = (i + 1) * 48

        if d in test_days:
            ax.axvspan(x0, x1, color=test_bg, alpha=0.6, zorder=0)
        else:
            ax.axvspan(x0, x1, color=train_bg, alpha=0.4, zorder=0)

    # Only two curves
    ax.plot(
        x,
        true_vals,
        color=true_color,
        linestyle="-",
        linewidth=1.8,
        label="True"
    )
    ax.plot(
        x,
        pred_vals,
        color=pred_color,
        linestyle="--",
        linewidth=1.8,
        label="Predicted"
    )

    # Day separators
    for start in day_starts[1:]:
        ax.axvline(start, color="gray", linewidth=0.8, alpha=0.35)

    ax.set_xticks(day_starts)
    ax.set_xticklabels(day_labels, rotation=45)
    ax.set_xlabel("Available days in month")
    ax.set_ylabel(target_name)
    ax.set_title(f"{target_name}: True vs predicted for {month}")

    # Legend for curves + sample type shading
    curve_handles, curve_labels = ax.get_legend_handles_labels()
    sample_handles = [
        mpatches.Patch(color=train_bg, alpha=0.8, label="In-sample day"),
        mpatches.Patch(color=test_bg, alpha=0.8, label="Out-of-sample day"),
    ]
    ax.legend(handles=curve_handles + sample_handles)

    ax.grid(True)
    plt.tight_layout()
    plt.show()

def plot_true_vs_predicted_all_months(
    final_result,
    target_name,
    true_color="#1b7f3b",
    pred_color="#c51b7d"
):
    """
    Generate the month plot for every available month.
    """
    train_days = list(final_result["y_train_daily"].index)
    test_days = list(final_result["y_test_daily"].index)

    all_days = sorted(set(train_days + test_days))
    months = sorted({pd.Timestamp(d).strftime("%Y-%m") for d in all_days})

    for month in months:
        plot_true_vs_predicted_month(
            final_result=final_result,
            target_name=target_name,
            month=month,
            true_color=true_color,
            pred_color=pred_color
        )
    
def print_final_train_test_metrics(final_result, target_name, include_month_cluster_summary=False):
    """
    Print in-sample vs out-of-sample performance for the final model.
    """
    train_metrics = final_result["model_result"]["train_metrics"]
    test_metrics = final_result["model_result"]["test_metrics"]

    print(target_name)
    print("TRAIN (in-sample)")
    print(f"  MAE: {train_metrics['mae']:.4f}")
    print(f"  RMSE: {train_metrics['rmse']:.4f}")
    print(f"  Daily MAE: {train_metrics['daily_mae']:.4f}")
    print(f"  Point coverage: {100 * train_metrics['point_coverage']:.1f}%")
    print(f"  Mean daily coverage: {100 * train_metrics['mean_daily_coverage']:.1f}%")

    print("TEST (out-of-sample)")
    print(f"  MAE: {test_metrics['mae']:.4f}")
    print(f"  RMSE: {test_metrics['rmse']:.4f}")
    print(f"  Daily MAE: {test_metrics['daily_mae']:.4f}")
    print(f"  Point coverage: {100 * test_metrics['point_coverage']:.1f}%")
    print(f"  Mean daily coverage: {100 * test_metrics['mean_daily_coverage']:.1f}%")

    if include_month_cluster_summary:
        month_cluster_mae = compute_month_cluster_mae(final_result)

        train_cells = month_cluster_mae[month_cluster_mae["dataset"] == "train"].shape[0]
        test_cells = month_cluster_mae[month_cluster_mae["dataset"] == "test"].shape[0]

        print("MONTH × CLUSTER DETAIL")
        print(f"  Train month-cluster cells: {train_cells}")
        print(f"  Test month-cluster cells: {test_cells}")


def plot_final_cluster_sizes(final_result):
    """
    Plot cluster sizes from the TRAIN portion of the final model.
    """
    cluster_sizes = final_result["model_result"]["cluster_size_info"]["cluster_sizes"]

    plt.figure(figsize=(8, 5))
    plt.bar(cluster_sizes.index.astype(str), cluster_sizes.values)
    plt.xlabel("Cluster")
    plt.ylabel("Number of training days")
    plt.title("Final model cluster sizes (TRAIN data only)")
    plt.grid(True, axis="y")
    plt.show()

    print("Smallest cluster size:", cluster_sizes.min())
    print("Largest cluster size:", cluster_sizes.max())

#CLUSTER x Month Metrics

def compute_month_cluster_mae(final_result):
    """
    Compute mean daily MAE per (month, cluster, dataset) for the final model.

    Returns
    -------
    month_cluster_mae : pd.DataFrame
        Columns: month, cluster, dataset, mean_mae, n_days
    """
    model_result = final_result["model_result"]

    rows = []

    # TRAIN
    y_true_train = final_result["y_train_daily"]
    y_pred_train = model_result["y_train_pred_daily"]
    train_clusters = model_result["train_cluster_labels"]

    for day, cluster in zip(y_true_train.index, train_clusters):
        month = pd.Timestamp(day).strftime("%Y-%m")
        mae_day = np.mean(np.abs(y_true_train.loc[day].values - y_pred_train.loc[day].values))

        rows.append({
            "month": month,
            "cluster": int(cluster),
            "dataset": "train",
            "mae": float(mae_day)
        })

    # TEST
    y_true_test = final_result["y_test_daily"]
    y_pred_test = model_result["y_test_pred_daily"]
    test_clusters = model_result["test_cluster_labels"]

    for day, cluster in zip(y_true_test.index, test_clusters):
        month = pd.Timestamp(day).strftime("%Y-%m")
        mae_day = np.mean(np.abs(y_true_test.loc[day].values - y_pred_test.loc[day].values))

        rows.append({
            "month": month,
            "cluster": int(cluster),
            "dataset": "test",
            "mae": float(mae_day)
        })

    df = pd.DataFrame(rows)

    month_cluster_mae = (
        df.groupby(["month", "cluster", "dataset"])
        .agg(
            mean_mae=("mae", "mean"),
            n_days=("mae", "size")
        )
        .reset_index()
        .sort_values(["month", "cluster", "dataset"])
    )

    return month_cluster_mae

    # Heatmaps!!

def plot_month_cluster_mae_heatmaps(month_cluster_mae, target_name):
    """
    Plot side-by-side heatmaps of month x cluster MAE for TRAIN and TEST.

    Parameters
    ----------
    month_cluster_mae : pd.DataFrame
        Output from compute_month_cluster_mae(...)
    target_name : str
        e.g. "PV_total" or "Wind_total"
    """
    train_df = month_cluster_mae[month_cluster_mae["dataset"] == "train"]
    test_df = month_cluster_mae[month_cluster_mae["dataset"] == "test"]

    train_pivot = train_df.pivot(index="month", columns="cluster", values="mean_mae")
    test_pivot = test_df.pivot(index="month", columns="cluster", values="mean_mae")

    # Align shapes so train/test use same months and clusters
    all_months = sorted(set(train_pivot.index).union(set(test_pivot.index)))
    all_clusters = sorted(set(train_pivot.columns).union(set(test_pivot.columns)))

    train_pivot = train_pivot.reindex(index=all_months, columns=all_clusters)
    test_pivot = test_pivot.reindex(index=all_months, columns=all_clusters)

    vmin = np.nanmin([train_pivot.to_numpy(), test_pivot.to_numpy()])
    vmax = np.nanmax([train_pivot.to_numpy(), test_pivot.to_numpy()])

    fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)

    im0 = axes[0].imshow(train_pivot, aspect="auto", vmin=vmin, vmax=vmax)
    axes[0].set_title(f"{target_name} month × cluster MAE (TRAIN)")
    axes[0].set_xlabel("Cluster")
    axes[0].set_ylabel("Month")
    axes[0].set_xticks(np.arange(len(all_clusters)))
    axes[0].set_xticklabels(all_clusters)
    axes[0].set_yticks(np.arange(len(all_months)))
    axes[0].set_yticklabels(all_months)

    im1 = axes[1].imshow(test_pivot, aspect="auto", vmin=vmin, vmax=vmax)
    axes[1].set_title(f"{target_name} month × cluster MAE (TEST)")
    axes[1].set_xlabel("Cluster")
    axes[1].set_ylabel("Month")
    axes[1].set_xticks(np.arange(len(all_clusters)))
    axes[1].set_xticklabels(all_clusters)
    axes[1].set_yticks(np.arange(len(all_months)))
    axes[1].set_yticklabels(all_months)

    # Optional: annotate cell values
    for ax, pivot in zip(axes, [train_pivot, test_pivot]):
        for i in range(pivot.shape[0]):
            for j in range(pivot.shape[1]):
                value = pivot.iloc[i, j]
                if pd.notna(value):
                    ax.text(j, i, f"{value:.2f}", ha="center", va="center", fontsize=8)

    cbar = fig.colorbar(im1, ax=axes, shrink=0.9)
    cbar.set_label("Mean daily MAE")

    plt.show()

def plot_month_cluster_count_heatmaps(month_cluster_mae, target_name):
    """
    Plot side-by-side heatmaps of number of days per month x cluster for TRAIN and TEST.
    """
    train_df = month_cluster_mae[month_cluster_mae["dataset"] == "train"]
    test_df = month_cluster_mae[month_cluster_mae["dataset"] == "test"]

    train_pivot = train_df.pivot(index="month", columns="cluster", values="n_days")
    test_pivot = test_df.pivot(index="month", columns="cluster", values="n_days")

    all_months = sorted(set(train_pivot.index).union(set(test_pivot.index)))
    all_clusters = sorted(set(train_pivot.columns).union(set(test_pivot.columns)))

    train_pivot = train_pivot.reindex(index=all_months, columns=all_clusters)
    test_pivot = test_pivot.reindex(index=all_months, columns=all_clusters)

    vmin = 0
    vmax = np.nanmax([train_pivot.to_numpy(), test_pivot.to_numpy()])

    fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)

    im0 = axes[0].imshow(train_pivot, aspect="auto", vmin=vmin, vmax=vmax)
    axes[0].set_title(f"{target_name} month × cluster counts (TRAIN)")
    axes[0].set_xlabel("Cluster")
    axes[0].set_ylabel("Month")
    axes[0].set_xticks(np.arange(len(all_clusters)))
    axes[0].set_xticklabels(all_clusters)
    axes[0].set_yticks(np.arange(len(all_months)))
    axes[0].set_yticklabels(all_months)

    im1 = axes[1].imshow(test_pivot, aspect="auto", vmin=vmin, vmax=vmax)
    axes[1].set_title(f"{target_name} month × cluster counts (TEST)")
    axes[1].set_xlabel("Cluster")
    axes[1].set_ylabel("Month")
    axes[1].set_xticks(np.arange(len(all_clusters)))
    axes[1].set_xticklabels(all_clusters)
    axes[1].set_yticks(np.arange(len(all_months)))
    axes[1].set_yticklabels(all_months)

    for ax, pivot in zip(axes, [train_pivot, test_pivot]):
        for i in range(pivot.shape[0]):
            for j in range(pivot.shape[1]):
                value = pivot.iloc[i, j]
                if pd.notna(value):
                    ax.text(j, i, f"{int(value)}", ha="center", va="center", fontsize=8)

    cbar = fig.colorbar(im1, ax=axes, shrink=0.9)
    cbar.set_label("Number of days")

    plt.show()

## $\text{\textcolor{green}{Cluster Models}}$

In [ ]:
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
# Read downloaded csv data and make df –– Last downloaded the power data in February 2026; I haven't used Jakob's new df

final_file = "data/complete_dataframe2_30min.csv"
df = pd.read_csv(str(ROOT/final_file), delimiter=',', decimal='.', parse_dates=['ts'], index_col='ts') #ts is already the index :)

# Power columns to aggregate
pv_cols = ['B117_m','B319_m','B330_1_m','B330_2_m','B330_3_m','B716_m','B715_m']
wind_power_cols = ['Aircon_WT Power_m', 'Gaia_WT Power_m']

df = aggregate_power_columns(df, pv_cols, wind_power_cols,drop_original=False)

print(df.columns)

# Example weather features
feature_cols_pv = [ "radia_glob", "temp_dry"]
feature_cols_wind = [ "wind_speed", "temp_dry"]


### $\text{\textcolor{green}{PV Cluster Model}}$

In [ ]:
# Prepare PV daily data
X_daily_pv, y_daily_pv, valid_days_pv = prepare_daily_profiles_single_output(
    df_input=df,
    feature_cols=feature_cols_pv,
    target_col="PV_total",
    samples_per_day=48
)

# Search best K for PV
pv_search_result = search_best_k_with_folds_single_output(
    X_daily=X_daily_pv,
    y_daily=y_daily_pv,
    valid_days=valid_days_pv,
    k_values= np.arange(2,18),
    n_folds=5,
    test_days_per_month=3,
    fold_random_state=42,
    model_random_state=42,
    min_allowed_cluster_size=5
)

print("Best K for PV:", pv_search_result["best_k"])
print(pv_search_result["summary"])


plot_mae_vs_k(pv_search_result["summary"], target_name="PV_total")
plot_min_cluster_size_vs_k(pv_search_result["summary"])
plot_coverage_vs_k(pv_search_result["summary"], target_name="PV_total")

Analysis points to a best k = 6 since it corresponds to the elbow point in the MAE vs K plot while remaining above a 3% treshold in the minimum cluster size.

In [ ]:
# Final PV model
pv_final_result = fit_final_model_single_output(
    X_daily=X_daily_pv,
    y_daily=y_daily_pv,
    valid_days=valid_days_pv,
    best_k=9, #Based on return of previous Kmeans analysis
    final_test_days_per_month=3,
    final_split_random_state=999,
    model_random_state=42
)

print_final_train_test_metrics(pv_final_result, target_name="PV_total")

#plot_cluster_spread_from_final_model(pv_final_result, wind_feature="wind_speed", radiation_feature="radia_glob")

plot_cluster_profiles_from_final_model(pv_final_result,target_name="PV_total", show_band=True)
plot_cluster_profiles_from_final_model(pv_final_result, target_name="PV total", show_band = False)

plot_true_vs_predicted_all_months(
    final_result=pv_final_result,
    target_name="PV_total"
)

plot_final_cluster_sizes(pv_final_result)

### $\text{\textcolor{green}{More Metrics -- Cluster X Month}}$

In [ ]:
pv_month_cluster_mae = compute_month_cluster_mae(pv_final_result)
plot_month_cluster_mae_heatmaps(pv_month_cluster_mae, target_name="PV_total")
plot_month_cluster_count_heatmaps(pv_month_cluster_mae, target_name="PV_total")

METRICS 

K = 5 NO wind_speed

PV_total
TRAIN (in-sample)
  MAE: 2.8155
  RMSE: 5.6274
  Daily MAE: 2.8155
  Point coverage: 86.0%
  Mean daily coverage: 86.0%
TEST (out-of-sample)
  MAE: 2.9795
  RMSE: 5.8144
  Daily MAE: 2.9795
  Point coverage: 82.5%
  Mean daily coverage: 82.5%

K = 5, With wind_speed
TRAIN (in-sample)
  MAE: 3.5492
  RMSE: 6.8970
  Daily MAE: 3.5492
  Point coverage: 85.1%
  Mean daily coverage: 85.1%
TEST (out-of-sample)
  MAE: 4.2044
  RMSE: 8.2225
  Daily MAE: 4.2044
  Point coverage: 82.4%
  Mean daily coverage: 82.4%

K = 12, with wind_speed, radia, no temp

PV_total
TRAIN (in-sample)
  MAE: 2.9090
  RMSE: 5.7651
  Daily MAE: 2.9090
  Point coverage: 83.8%
  Mean daily coverage: 83.8%
TEST (out-of-sample)
  MAE: 3.5952
  RMSE: 7.5471
  Daily MAE: 3.5952
  Point coverage: 78.2%
  Mean daily coverage: 78.2%


### $\text{\textcolor{green}{Wind Cluster Model }}$

In [ ]:
# Prepare Wind daily data
X_daily_wind, y_daily_wind, valid_days_wind = prepare_daily_profiles_single_output(
    df_input=df,
    feature_cols=feature_cols_wind,
    target_col="Wind_total",
    samples_per_day=48
)

# Search best K for Wind
wind_search_result = search_best_k_with_folds_single_output(
    X_daily=X_daily_wind,
    y_daily=y_daily_wind,
    valid_days=valid_days_wind,
    k_values=np.arange(2,18),
    n_folds=5,
    test_days_per_month=3,
    fold_random_state=42,
    model_random_state=42,
    min_allowed_cluster_size=5
)

print("Best K for Wind:", wind_search_result["best_k"])
print(wind_search_result["summary"])

plot_mae_vs_k(wind_search_result["summary"], target_name="Wind_total")
plot_min_cluster_size_vs_k(wind_search_result["summary"])
plot_coverage_vs_k(wind_search_result["summary"], target_name="Wind_total")

In [ ]:
# Final Wind model
wind_final_result = fit_final_model_single_output(
    X_daily=X_daily_wind,
    y_daily=y_daily_wind,
    valid_days=valid_days_wind,
    best_k= 10, #From search of best K
    final_test_days_per_month=3,
    final_split_random_state=999,
    model_random_state=42
)

print_final_train_test_metrics(wind_final_result, target_name="Wind_total")

#plot_cluster_spread_from_final_model(wind_final_result,wind_feature="wind_speed",radiation_feature="radia_glob")

plot_cluster_profiles_from_final_model(wind_final_result,target_name="Wind_total", show_band=True)
plot_cluster_profiles_from_final_model(wind_final_result, target_name="Wind total", show_band = False)

plot_true_vs_predicted_all_months(
    final_result=wind_final_result,
    target_name="Wind_total"
)

plot_final_cluster_sizes(wind_final_result)

## K = 5 (yes radiation, wind_speed, temp, wind_max)

- Wind_total
- TRAIN (in-sample)
  - MAE: 2.0693
  - RMSE: 3.1175
  - Daily MAE: 2.0693
  - Point coverage: 84.6%
  - Mean daily coverage: 84.6%
- TEST (out-of-sample)
  - MAE: 2.5442
  - RMSE: 3.7145
  - Daily MAE: 2.5442
  - Point coverage: 77.2%
  - Mean daily coverage: 77.2%

## K = 10 (yes radiation, wind_speed, temp, wind_max)

- Wind_total
- TRAIN (in-sample)
  - MAE: 1.8740
  - RMSE: 2.8830
  - Daily MAE: 1.8740
  - Point coverage: 83.6%
  - Mean daily coverage: 83.6%
- TEST (out-of-sample)
  - MAE: 2.5802
  - RMSE: 3.8449
  - Daily MAE: 2.5802
  - Point coverage: 74.9%
  - Mean daily coverage: 74.9%

## K = 10 (no radiation, wind_max yes wind_speed, temp)

- Wind_total
- TRAIN (in-sample)
  - MAE: 1.8721
  - RMSE: 2.9645
  - Daily MAE: 1.8721
  - Point coverage: 83.8%
  - Mean daily coverage: 83.8%
- TEST (out-of-sample)
  - MAE: 1.7999
  - RMSE: 2.8949
  - Daily MAE: 1.7999
  - Point coverage: 81.5%
  - Mean daily coverage: 81.5%

- Wind_total
- TRAIN (in-sample)
  - MAE: 1.9975
  - RMSE: 3.0252
  - Daily MAE: 1.9975
  - Point coverage: 83.7%
  - Mean daily coverage: 83.7%
- TEST (out-of-sample)
  - MAE: 1.6410
  - RMSE: 2.5180
  - Daily MAE: 1.6410
  - Point coverage: 88.4%
  - Mean daily coverage: 88.4%